## Abstract

In this computational study, a helix penalty shifts ESMFold2 binder designs toward β-rich structures across five targets, with lower predicted confidence. On PD-L1, steering works best early in optimization, and changing the initial soft amino-acid distribution provides a second way to redirect composition without a helicity penalty. Initialization affects the final designs, but loss steering yields more candidates meeting the combined sheet and confidence criteria. We map these controls and their costs to guide experimental testing.

----

:::{.callout-note title="AI usage disclosure" collapse="true"}
AI assistance was used to review the analysis and revise this draft. The author-approved disclosure will be added before publication.
:::

## Purpose

We wanted to make ESMFold2 produce more β-rich binder candidates without retraining the model. Its unmodified design procedure returned mostly helical structures in our campaigns, which limited the secondary-structure compositions available for screening.

We added a penalty for helix-like geometry and measured its effects across five targets. We then varied when the penalty acted during optimization on PD-L1 and tested whether changing the initial soft sequence could redirect composition under the original objective. This notebook reports how these two controls change secondary structure and predicted confidence, and which conclusions still require experimental validation.

## Introduction

ESMFold2 is a structure prediction model that uses representations from the ESMC protein language model to predict proteins and their complexes. Alongside the model, @candido2026language present a binder-design algorithm that combines structural feedback from ESMFold2 with sequence feedback from ESMC. Starting from a soft sequence---a probability distribution over amino acids at each position---the algorithm repeatedly predicts the binder–target complex, evaluates a design objective, and updates the candidate sequence. As optimization proceeds, the soft sequence is progressively hardened into a discrete amino-acid sequence. The weights of both ESMC and ESMFold2 remain frozen, allowing researchers to design proteins under different objectives without retraining or fine-tuning either model. This approach builds on earlier work in protein hallucination and gradient-based sequence optimization [@anishchenko2021hallucination; @norn2021landscape; @frank2024relaxed], and was popularized for binder design by BindCraft [@pacesa2025bindcraft].

Although training data, model architectures, design objectives, and filtering choices differ among methods, a tendency toward α-helical designs has been reported in distinct binder-design workflows. BindCraft reports this tendency in its confidence-driven designs [@pacesa2025bindcraft], and RFdiffusion's developers note that the default model often generates predominantly helical binders [@rosettacommons2025rfdiffusiondocs]. A separate ten-target benchmark reports high helix content among RFdiffusion designs passing its computational screen, while PXDesign-d produces a broader range of helix fractions [@protenix2025pxdesign]. We observed a similar enrichment in unsteered ESMFold2 designs, motivating us to investigate how this preference could be changed.

Reducing this preference has several potential benefits: bringing designed secondary-structure compositions closer to those of natural binding proteins, increasing structural diversity within a campaign, and reaching binding sites that favor different interface geometries. The last motivation has experimental support. Guiding RFdiffusion toward β-strand pairing produced binders to exposed sheet edges, including on ALK-3 and KIT, where earlier design campaigns had failed [@sappington2026betapairing]. Such results motivate exploring alternatives to predominantly helical designs.

BindCraft already uses a helicity loss to obtain β-rich designs, with a decrease in computational screening success [@pacesa2025bindcraft]. To our knowledge, secondary-structure steering with a helicity penalty has not been demonstrated for ESMFold2. Understanding where the preference arises could also reveal other ways to intervene: a helical final design may reflect a preference already present in the initial prediction, its reinforcement during sequence optimization, or both.

Here, we investigate how to redirect ESMFold2's helix preference and what that costs in predicted confidence. We vary the strength of a helicity penalty across targets and use PD-L1 scheduling experiments to determine when it acts most effectively. The penalty increases sheet content and the number of candidates passing a combined sheet-content and confidence screen, despite lowering average confidence. On PD-L1, we also detect a contribution to helicity before the first sequence update and show that changing the initial soft amino-acid probabilities can alter composition through full optimization without a helicity penalty. Together, these analyses establish the design objective and sequence initialization as two controls over the resulting composition, with explicit loss steering yielding more candidates that pass our in silico screen.

:::{.callout-note collapse="true" title="GPU computation and reproducibility"}

Publication execution measures preserved model structures, scores and traces without running GPU models. The GPU outputs have not yet been publicly archived; reproduction currently requires those local inputs. Target inputs and the generation code are included in this repository.

New designs can be generated explicitly using `python -m analysis.campaign` after deploying the Modal workers. GPU runs cost compute and are not seed-reproducible, so new runs are new samples rather than replacements for the reported designs. See the reproduction guide for input locations and commands.

:::


In [ ]:
# | include: false

import warnings

import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)

In [ ]:
# | include: false
from analysis import (
    composition,
    deposit,
    figures,
    loss_trace,
    paths,
    reference,
    targets,
)

paths.ensure_dirs()

TARGETS = list(targets.TARGET_SPECS)
ARMS = list(targets.CAMPAIGN_ARMS)
SCAN_TARGET = "pdl1_igv"
ONSET_ARMS = list(dict.fromkeys(targets.ONSET_SCAN.values()))
RELEASE_ARMS = list(dict.fromkeys(targets.RELEASE_SCAN.values()))
SCAN_ARMS = sorted(set(ONSET_ARMS) | set(RELEASE_ARMS))

if any(not composition.is_available(t, a) for t, a in targets.publication_campaigns()):
    deposit.DESIGN_TABLES.fetch()

## Unsteered designs are helix-rich relative to natural proteins

We generated 30 unsteered binders per target using the five constructs from the ESMFold2 paper [@candido2026language] (@tbl-targets). Binder lengths were sampled from a 60–200-residue range.

| Target | Target region | UniProt ID | UniProt residues | Target length (aa) |
|---|---|---|---|---:|
| EGFR | Domain III | [P00533](https://www.uniprot.org/uniprotkb/P00533/entry) | 334–524 | 191 |
| PDGFRβ | Domains D2–D3 | [P09619](https://www.uniprot.org/uniprotkb/P09619/entry) | 126–312 | 187 |
| PD-L1 | IgV domain | [Q9NZQ7](https://www.uniprot.org/uniprotkb/Q9NZQ7/entry) | 18–132 | 115 |
| CTLA-4 | IgV domain | [P16410](https://www.uniprot.org/uniprotkb/P16410/entry) | 38–155 | 118 |
| CD45 | Domains d3–d4 | [P08575](https://www.uniprot.org/uniprotkb/P08575/entry) | 390–574 | 185 |

: Human target constructs from Table S10 of @candido2026language. Residue ranges use canonical UniProt numbering and include both endpoints. {#tbl-targets}

To compare their secondary-structure composition with binders found in nature, we queried the RCSB Protein Data Bank for experimentally resolved complexes containing two distinct protein entities and selected chains within the same 60–200-residue range. We refer to each selected chain as the natural binder and the other as its partner.

We then compared the helix, sheet, and loop fractions of the two groups within five binder-length bins (@fig-composition).

:::{.callout-note collapse="true" title="Natural protein baseline construction"}

We selected X-ray structures containing two distinct protein entities and a binder with a named source organism other than "synthetic construct," retaining one representative per sequence-identity cluster. We excluded records annotated as designed proteins and chains outside the length range or with insufficient resolved backbone. The code below loads the saved measurements, rebuilding them from archived RCSB inputs if needed, and reports the query settings and retained dataset.

Both natural and designed chains are measured with PyDSSP using the same backbone-based helix, sheet, and loop assignments [@minami2024pydssp]. Only residues with N, Cα, C, and O coordinates contribute to the fractions.

In [ ]:
# | code-fold: true
# | code-summary: "Load and summarize the natural protein baseline"

import json

from analysis import rcsb

natural = reference.composition()
designs = composition.load_all(TARGETS, ["vanilla"])

# Read the recorded query rather than issuing a new search against the live PDB.
query = json.loads(paths.reference_table(reference.QUERY_RECORD).read_text())
criteria = {
    node["parameters"]["attribute"]: node["parameters"].get("value")
    for node in query["query"]["nodes"]
}
length = criteria["entity_poly.rcsb_sample_sequence_length"]
resolution = criteria["rcsb_entry_info.resolution_combined"]
released = criteria["rcsb_accession_info.initial_release_date"]
identity = query["request_options"]["group_by"]["similarity_cutoff"]

print(f"RCSB query: X-ray resolution ≤ {resolution:g} Å; released by {released}")
print(f"Binder length: {length['from']}–{length['to']} residues")
print(f"Sequence-identity clustering: {identity}%")
print(f"Minimum resolved backbone fraction: {rcsb.MIN_OBSERVED_FRACTION:.0%}")
print(f"Retained reference: {len(natural):,} chains from {natural.pdb_id.nunique():,} PDB entries")
print(f"  Small-interface flag: {int(natural.flag_small_interface.sum()):,} chains")
print(f"  Peptide-partner flag: {int(natural.flag_peptide_target.sum()):,} chains")

:::

In [ ]:
# | label: fig-composition
# | fig-cap: "Secondary-structure composition of unsteered designs and the experimental reference, grouped into five binder-length bins. Unfilled bars show the reference and filled bars show designs. Each chain contributes once to its bin. Bars are means; whiskers are standard errors across chains. Panel y-axes differ."
# | fig-subcap:
# |   - "Helix"
# |   - "Beta sheet"
# |   - "Loop"
# | layout-ncol: 2
# | code-fold: true
# | code-summary: "Plot code"

for state in ("helix", "sheet", "loop"):
    figures.composition_bars(natural, designs, state)

Unsteered ESMFold2 designs are substantially more helical than the natural reference, with correspondingly less sheet and loop (@fig-composition). This pattern recurs across binder lengths and all five targets, suggesting a persistent preference in the design workflow. Redirecting that preference could help a design campaign explore more of the secondary-structure compositions found among natural binding partners.

## A helix penalty changes the design objective

The reference comparison suggests a practical opportunity: reduce the predominance of helices so that a campaign samples more of the sheet-containing compositions found in natural complexes. To test this, we added a term to the existing objective that penalizes helix-like geometry.

In [ ]:
# | include: false

steered = composition.load_all(TARGETS, ARMS)
trace = loss_trace.load_all(TARGETS, ARMS)

TARGET_ORDER = (
    steered[steered["arm"] == "helix_mean_w05"]
    .groupby("target")["helix (fraction 0-1)"]
    .median()
    .sort_values()
    .index.tolist()
)
STEERED_ARMS = [arm for arm in ARMS if arm != "vanilla"]

onset = composition.load_all([SCAN_TARGET], ONSET_ARMS)
onset_trace = loss_trace.load_all([SCAN_TARGET], ONSET_ARMS)
release = composition.load_all([SCAN_TARGET], RELEASE_ARMS)
release_trace = loss_trace.load_all([SCAN_TARGET], RELEASE_ARMS)

### The existing structural objective

The default structural loss combines intrabinder contacts, contacts between the target and binder, and a size-dependent compactness term:

$$
\mathcal{L}_{\mathrm{struct}}
= 0.5\,\mathcal{L}_{\mathrm{intra}}
+ 0.5\,\mathcal{L}_{\mathrm{inter}}
+ 0.2\,\mathcal{L}_{\mathrm{glob}}.
$$

The intrabinder term favors nonlocal contacts within the binder. The interface term asks each target residue for a contact with the binder, and the compactness term favors smaller predicted binder dimensions. For details, see @candido2026language.

### Penalizing helix-like distances

We use a local geometric feature of an α helix: residues four positions apart have nearby Cα atoms. For each eligible binder position, we sum the distogram probability below 7 Å and average over positions:

$$
\mathcal{L}_{\mathrm{helix}}=\frac{1}{L-4}\sum_{i=1}^{L-4}
P(d_{i,i+4}<7\ \text{Å})\in[0,1].
$$

The final four residues have no partner at this offset and are excluded from the average. The penalty acts only on the binder block of the distogram.

To steer sequence optimization, we replace the default structural objective $\mathcal{L}_{\mathrm{struct}}$ with an objective that includes the weighted helicity penalty:

$$
\mathcal{L}^{\prime}_{\mathrm{struct}}
=\mathcal{L}_{\mathrm{struct}}+w\,\mathcal{L}_{\mathrm{helix}}.
$$

Increasing the weight $w$ makes these short $i,i+4$ distances more costly.

The term $\mathcal{L}_{\mathrm{helix}}$ is a differentiable proxy, not a secondary-structure assignment. A low value does not distinguish a beta strand from an irregular extended segment, and it does not specify how strands should pair. We therefore evaluate the resulting structures separately with PyDSSP. @fig-scene illustrates the distance criterion on an example protein.

:::{.callout-note collapse="true" title="Relationship to BindCraft's helicity loss"}

We developed this formulation before becoming aware of BindCraft's helicity loss [@pacesa2025bindcraft]. Both use the same underlying strategy of steering secondary structure by discouraging short intrabinder distances characteristic of helices. BindCraft's [implementation](https://github.com/martinpacesa/BindCraft/blob/7cd4ace1b7407adf66a50dfefa47de2270f5e4a9/functions/colabdesign_utils.py) averages a [negative-log contact probability](https://github.com/sokrypton/ColabDesign/blob/24a449493c9e60fc8fefd4981a4b83f7900b9110/colabdesign/af/loss.py) for binder residue pairs separated by three positions, using a 6Å cutoff. Our formulation averages contact probabilities for pairs separated by four positions, using a 7Å cutoff. We have not made a direct comparison of the two formulations under matched conditions.
:::

In [ ]:
# | label: fig-scene
# | fig-cap: "A predicted binder colored by the coordinate-based distance criterion underlying the penalty. Red marks an i-to-i+4 Cα pair below 7 Å; each such pair contributes 1/(L-4) to the binary coordinate average. The last four residues are unscored. Hovering shows the measured pair and distance. Keep in mind that the optimized loss uses distogram probabilities rather than realized coordinate values."
# | code-fold: true
# | code-summary: "Plot code"

from analysis import scene

scene.loss_term_scene(paths.ASSETS / "example-complex.pdb").show()

### Increasing the weight replaces helix with sheet and loop

We compared weights of 0, 0.5, 1.0, 2.0, and 5.0, with 30 seeds per condition on each target. @fig-scan-1 shows the mean composition pooled across all five targets; @fig-scan-2 shows the pooled distribution of helix fractions.

In [ ]:
# | label: fig-scan
# | fig-cap: "Secondary-structure composition across steering weights. The stacked bars pool all five targets, with 150 designs per weight and equal contributions from each target; the reference includes all natural chains without resampling or weighting. The violins show smoothed helix-fraction distributions for the same 150 designs per weight, restricted to each group’s observed range and scaled to equal maximum width. White dots mark medians; thick vertical lines span the interquartile range. PyDSSP assigns secondary structure from backbone coordinates: the primary critic’s final predicted complexes for designs and experimental structures for the natural reference."
# | fig-subcap:
# |   - "Mean PyDSSP helix, sheet, and loop fractions from final predicted binder structures across all five targets (150 designs per weight). The natural reference uses PyDSSP assignments from experimental structures."
# |   - "Distribution of PyDSSP helix fractions from final predicted binder structures across all five targets (150 designs per weight). Dots mark medians; lines span the interquartile range."
# | layout-ncol: 2
# | code-fold: true
# | code-summary: "Plot code"

from matplotlib import pyplot as plt

figures.composition_stack(steered, None, ARMS, natural)
figures.scan_violin(steered, ARMS, "helix")
plt.show()

@fig-scan-1 shows that penalizing helix-like distances increases both sheet and loop content. At the strongest weights, helices are almost absent, while sheets account for just over half of the mean composition.

The pooled distributions in @fig-scan-2 reveal heterogeneous responses to intermediate steering weights. In particular, at $w=0.5$, the helix-fraction distribution appears bimodal, with some designs remaining strongly helical, while others contain little helix. This pattern suggests that modest steering can substantially redirect some optimization trajectories while leaving others helix-rich. Both helix-rich and helix-poor designs also occur within individual targets (@fig-generality).

### The response occurs on all five targets

We next examined how individual targets contribute to the pooled response. @fig-generality shows the distributions of helix, sheet, and loop fractions for every target and weight.

In [ ]:
# | label: fig-generality
# | fig-cap: "Final secondary-structure fractions across all five targets and steering weights (30 designs per target and weight), assigned by PyDSSP from the primary critic’s predicted binder structures. Rows show helix, sheet, and loop fractions; columns show targets, with shared axes. Points show individual designs with horizontal jitter; boxes span the interquartile range, central lines mark medians, and whiskers extend to the most extreme observations within 1.5 times the interquartile range. Zero denotes unsteered design. Conditions are equally spaced; spacing does not represent the numerical difference between weights."
# | out-width: "100%"
# | code-fold: true
# | code-summary: "Plot code"

from matplotlib import pyplot as plt

figures.scan_boxplots(steered, TARGET_ORDER, ARMS)
plt.show()

Median helix decreases with weight on all five targets, with some targets being more responsive to intermediate weights (e.g. CD45) than others (e.g. PD-L1). Yet all reach a median helix fraction of zero by weight 5.0. Commensurately, beta sheet and loop proportions increase, indicating a shift toward sheet-rich designs with substantial loop content. The broad distributions at intermediate weights show that a given steering weight can still yield very different compositions that depends on the initial seed.

### Steering lowers confidence

Steering produces more sheet-rich designs, but does it retain enough high-confidence candidates to make that change useful for a design campaign? We assessed confidence in the binder–target arrangement using *interface predicted TM-score* (ipTM), and confidence in local binder structure using binder *predicted local distance difference test score* (pLDDT). @fig-cost and @tbl-cost show how confidence changes across the full population of designs. We then ask how many candidates satisfy confidence thresholds, and how many also meet the desired sheet content.

In [ ]:
# | label: fig-cost
# | fig-cap: "Primary-critic ipTM (top) and binder pLDDT (bottom) across steering weights, with 30 designs per target and weight. Columns use the same target order as @fig-generality. Points show individual designs with horizontal jitter; boxes span the interquartile range, central lines mark medians, and whiskers extend to the most extreme observations within 1.5 times the interquartile range. Box colors identify targets. Y-axis limits are shared within each row and span the observed values; neither row starts at zero. Zero on the x-axis denotes unsteered design. Conditions are equally spaced; spacing does not represent the numerical difference between weights."
# | out-width: "100%"
# | code-fold: true
# | code-summary: "Plot code"

from matplotlib import pyplot as plt

figures.scan_boxplots(steered, TARGET_ORDER, ARMS, ("iptm", "plddt"))
plt.show()

In [ ]:
# | label: tbl-cost
# | tbl-cap: "Median primary-critic ipTM and binder pLDDT by target and steering weight, each based on 30 designs. ipTM is on a 0–1 scale; binder pLDDT is on a 0–100 scale."
# | code-fold: true
# | code-summary: "Table code"

from IPython.display import HTML, display

cost_metric_labels = {
    "iptm (0-1)": "ipTM",
    "binder pLDDT (0-100)": "Binder pLDDT",
}
cost_long = steered.melt(
    id_vars=["target", "arm"],
    value_vars=list(cost_metric_labels),
    var_name="Metric",
    value_name="Score",
)
cost_long["Target"] = cost_long["target"].map(figures.TARGET_LABELS)
cost_long["Metric"] = cost_long["Metric"].map(cost_metric_labels)
cost_row_order = pd.MultiIndex.from_product(
    [cost_metric_labels.values(), [figures.TARGET_LABELS[target] for target in TARGET_ORDER]],
    names=["Metric", "Target"],
)
cost_table = (
    cost_long.pivot_table(
        index=["Metric", "Target"], columns="arm", values="Score", aggfunc="median"
    )
    .reindex(index=cost_row_order, columns=ARMS)
    .rename(columns={**figures.ARM_LABELS, "vanilla": "0 (unsteered)"})
    .rename_axis(columns=None)
)
cost_table.columns = pd.MultiIndex.from_product([["Steering weight"], cost_table.columns])
cost_group_styles = [
    {
        "selector": f"tbody tr:nth-child({group_number * len(TARGET_ORDER) + 1}) > *",
        "props": [("border-top", "1px solid #bfc1c3")],
    }
    for group_number in range(len(cost_metric_labels))
]

display(
    HTML(
        cost_table.style.set_table_styles(
            [
                {"selector": "", "props": [("border-collapse", "collapse"), ("width", "100%")]},
                {
                    "selector": "th, td",
                    "props": [
                        ("padding", "0.35rem 0.75rem"),
                        ("white-space", "nowrap"),
                    ],
                },
                {
                    "selector": "td, th.col_heading",
                    "props": [
                        ("min-width", "4rem"),
                        ("text-align", "center"),
                        ("font-variant-numeric", "tabular-nums"),
                    ],
                },
                {
                    "selector": "th.col_heading.level0",
                    "props": [
                        ("padding-bottom", "0.5rem"),
                        ("border-bottom", "1px solid #bfc1c3"),
                    ],
                },
                {
                    "selector": "th.row_heading, th.index_name",
                    "props": [
                        ("text-align", "left"),
                        ("vertical-align", "middle"),
                    ],
                },
            ]
        )
        .set_table_styles(cost_group_styles, overwrite=False)
        .format("{:.3f}", subset=pd.IndexSlice[("ipTM", slice(None)), :])
        .format("{:.1f}", subset=pd.IndexSlice[("Binder pLDDT", slice(None)), :])
        .to_html()
    )
)

Both confidence measures generally decline as steering increases, although the size of the decline depends on the target. EGFR and PD-L1 retain relatively high interface confidence, while CD45 shows a more pronounced reduction. The distributions nevertheless retain candidates with high scores even at strong steering weights, meaning averages are insufficient to tell us how many candidates may be viable for selection.

To distinguish these outcomes, we apply two *in silico* screens (@tbl-screen-criteria). First, we introduced a general quality screen based on pLDDT and ipTM thresholds. We then complemented this screen by tacking on the further requirement that designs must have at least be composed of a certain fractino of predicted beta sheet. The thresholds for both screens are shown below.

| Screen | Requirements |
|---|---|
| Quality | ipTM > 0.85 and pLDDT > 80 |
| Quality + sheet content | Quality criteria and sheet fraction > 0.40 |

: Criteria defining a candidate that passes each in silico screen. {#tbl-screen-criteria}

@tbl-sheet-screen reports the number of candidates passing each screen, so the loss of high-confidence candidates can be compared directly with the yield of candidates that also have the desired composition.

In [ ]:
# | label: tbl-sheet-screen
# | tbl-cap: "Candidates passing each in silico screen by target and steering weight. Each target has 30 designs per weight, for 150 designs per column. Pass rates use all designs at that weight as the denominator. Confidence scores and PyDSSP secondary structure come from the primary critic."
# | code-fold: true
# | code-summary: "Table code"

quality_pass = (steered["iptm (0-1)"] > composition.MIN_IPTM) & (
    steered["binder pLDDT (0-100)"] > composition.MIN_BINDER_PLDDT
)
screen_labels = ["Quality", "Quality + sheet content"]
screen_data = steered[["target", "arm"]].assign(
    **{
        screen_labels[0]: quality_pass,
        screen_labels[1]: composition.viable(steered),
    }
)
screen_denominators = screen_data.groupby("arm").size().reindex(ARMS)
screen_tables = {}
for screen in screen_labels:
    counts = (
        screen_data.groupby(["target", "arm"])[screen]
        .sum()
        .unstack("arm")
        .reindex(index=TARGET_ORDER, columns=ARMS)
        .rename(index=figures.TARGET_LABELS)
    )
    counts.loc["Total"] = counts.sum()
    counts.loc["Pass rate"] = counts.loc["Total"] / screen_denominators
    screen_tables[screen] = counts

screen_table = (
    pd.concat(screen_tables, names=["Screen", "Target"])
    .rename(columns={**figures.ARM_LABELS, "vanilla": "0 (unsteered)"})
    .rename_axis(columns=None)
)
screen_table.columns = pd.MultiIndex.from_product([["Steering weight"], screen_table.columns])
screen_group_styles = []
screen_group_start = 1
for screen in screen_labels:
    screen_group_styles.append(
        {
            "selector": f"tbody tr:nth-child({screen_group_start}) > *",
            "props": [("border-top", "1px solid #bfc1c3")],
        }
    )
    screen_group_start += len(screen_tables[screen])

display(
    HTML(
        screen_table.style.set_table_styles(
            [
                {"selector": "", "props": [("border-collapse", "collapse"), ("width", "100%")]},
                {
                    "selector": "th, td",
                    "props": [
                        ("padding", "0.35rem 0.75rem"),
                        ("white-space", "nowrap"),
                    ],
                },
                {
                    "selector": "td, th.col_heading",
                    "props": [
                        ("min-width", "4rem"),
                        ("text-align", "center"),
                        ("font-variant-numeric", "tabular-nums"),
                    ],
                },
                {
                    "selector": "th.col_heading.level0",
                    "props": [
                        ("padding-bottom", "0.5rem"),
                        ("border-bottom", "1px solid #bfc1c3"),
                    ],
                },
                {
                    "selector": "th.row_heading, th.index_name",
                    "props": [
                        ("text-align", "left"),
                        ("vertical-align", "middle"),
                    ],
                },
                {
                    "selector": "th.row_heading.level0",
                    "props": [
                        ("white-space", "normal"),
                        ("max-width", "12rem"),
                    ],
                },
            ]
        )
        .set_table_styles(screen_group_styles, overwrite=False)
        .format("{:.0f}")
        .format("{:.1%}", subset=pd.IndexSlice[(slice(None), "Pass rate"), :])
        .set_properties(
            subset=pd.IndexSlice[(slice(None), ["Total", "Pass rate"]), :],
            **{"font-weight": "bold"},
        )
        .to_html()
    )
)

Steering reduces the pooled number of candidates passing the quality screen, yet every tested steering weight yields more candidates passing the combined screen than unsteered design. At weight 2.0, quality passes fall from 103 to 30, while quality + sheet content passes rise from 15 to 29. Nearly all candidates that retain high confidence at this weight also meet the sheet requirement. Steering therefore trades a larger pool of high-confidence candidates for a smaller pool enriched in the desired composition, increasing the absolute number that satisfy both requirements.

Overall, steering away from ESMFold2's helix preference comes at the cost of lower in silico confidence. However, the biological meaning of that reduction remains unresolved because the same model family guides sequence optimization and evaluates the resulting designs. Lower scores could indicate poorer structures or weaker binding, but they could also reflect less reliable confidence estimates for sheet-rich designs. The model's preference for generating helix-rich designs motivates this possibility, although it doesn't establish that their confidence scores are better calibrated.

## Steering is most effective early in optimization

The final structures show how much the penalty changes a design campaign, but they don't reveal how that change develops. The soft sequence could become less helical gradually, or its composition could diverge from the unsteered trajectory early and then persist. Following the helix proxy during optimization lets us distinguish these patterns and identify when the sequence responds most strongly to the added objective. We also track the interface-confidence proxy to see how its development relates to the composition response.

In [ ]:
# | label: fig-trace
# | fig-cap: "Median optimization diagnostics across 150 designs per condition, pooling five targets. The helix proxy is the mean predicted short-distance probability; the interface proxy is also computed from the distogram. Neither is the final primary-critic readout."
# | fig-subcap:
# |   - "The loss term's own value"
# |   - "Distogram interface pTM proxy"
# | layout-ncol: 2
# | code-fold: true
# | code-summary: "Plot code"

for metric in ("helix_mean", "iptm_proxy"):
    figures.trace_lines(trace, metric, ARMS)

Helicity responds early to steering, while interface confidence continues developing later in optimization (@fig-trace). Within the first ten steps, the helix proxy rises in unsteered designs and falls sharply under stronger penalties. Those conditions remain separated through the rest of the run, even as the interface-confidence proxy continues to improve.

The weaker penalty at weight 0.5 initially lowers $\mathcal{L}_{\mathrm{helix}}$, but the proxy subsequently rises again. Early suppression is therefore not always sustained. Together, these trajectories motivate testing both when steering must begin and how long it must remain active.

:::{.callout-note collapse="true" title="Individual trajectories show the range of responses"}

The median trajectories in @fig-trace summarize runs with different outcomes. For example, the thirty CD45 runs at weight 1.0 include both low- and high-helicity endpoints (@fig-seeds).

In [ ]:
# | label: fig-seeds
# | fig-cap: "Distogram helicity trajectories for all thirty CD45 runs at weight 1.0. Thin lines show individual seeds and the thick line shows their median. These values are the optimization proxy, not final PyDSSP helix fractions."
# | code-fold: true
# | code-summary: "Plot code"

figures.seed_traces(
    trace[trace["target"] == "cd45_d3d4"], "cd45_d3d4", "helix_mean_w10", "helix_mean"
);

Seventeen of thirty CD45 runs finish with the helix proxy below 0.15, while the largest endpoint is 0.805. Their median endpoint of 0.098 summarizes this range. Individual traces provide a more resolved view of the response to the same target and penalty weight.

:::

### Penalty timing shapes composition and confidence

The trajectories show an early separation between steered and unsteered runs, followed by continued improvement in the interface proxy. This raises two questions: can a penalty introduced later still redirect composition, and can early steering be withdrawn while preserving that change and recovering confidence? We tested both on PD-L1 using weight 2.0 as a strong perturbation with a clear composition response across the target panel. The onset scan varies when the penalty begins; the withdrawal scan applies it from step 0 and varies when it ends. Each condition contains 30 designs and runs for 150 steps, with unsteered and continuously steered campaigns anchoring both scans.

Individual designs are not seed-reproducible in this workflow. These campaigns therefore compare outcome distributions under different schedules, rather than tracking the same design with and without a change in timing.

In [ ]:
# | label: fig-schedule
# | fig-cap: "PD-L1 onset and withdrawal scans at weight 2.0, with 30 designs per condition and 150 optimization steps. Upper panels show median distogram helicity, an optimization proxy. Final composition values reported in the text are separate PyDSSP measurements of the selected, refolded designs. Each lower panel shows final primary-critic ipTM above binder pLDDT for those designs. Points show individual designs with horizontal jitter; boxes span the interquartile range, central lines mark medians, and whiskers extend to the most extreme observations within 1.5 times the interquartile range. Purple boxes identify PD-L1 designs and gray boxes the unsteered baseline. Confidence-axis limits are shared across schedules for each metric. Onset step k applies the penalty from step k; withdrawal step k removes it before step k. The endpoints reuse the unsteered and continuously steered campaigns. Lower-panel conditions are equally spaced, not positioned in proportion to step number. Never denotes no onset in the left column and no withdrawal in the right column."
# | fig-subcap:
# |   - "Loss term against step, weight arriving at the labelled step"
# |   - "Loss term against step, weight withdrawn at the labelled step"
# |   - "Final confidence by penalty onset"
# |   - "Final confidence by penalty withdrawal"
# | layout-ncol: 2
# | code-fold: true
# | code-summary: "Plot code"

schedule_designs = pd.concat([onset, release], ignore_index=True)
schedule_confidence_limits = {
    metric: figures.confidence_limits(
        schedule_designs, metric, [SCAN_TARGET], list(schedule_designs["arm"].unique())
    )
    for metric in ("iptm", "plddt")
}

figures.schedule_traces(
    onset_trace,
    targets.ONSET_SCAN,
    (targets.SGD_STEPS, 0, 1, 2, 4, 7, 10),
    "helix_mean",
    figures.ONSET_NOTATION,
    targets.SGD_STEPS,
    targets.SGD_STEPS,
)
plt.show()
figures.schedule_traces(
    release_trace,
    targets.RELEASE_SCAN,
    (0, 1, 2, 4, 7, 10, 20, targets.SGD_STEPS),
    "helix_mean",
    figures.RELEASE_NOTATION,
    0,
    targets.SGD_STEPS,
)
plt.show()
figures.schedule_confidence(
    onset,
    targets.ONSET_SCAN,
    "Penalty onset step",
    targets.SGD_STEPS,
    targets.SGD_STEPS,
    limits=schedule_confidence_limits,
)
plt.show()
figures.schedule_confidence(
    release,
    targets.RELEASE_SCAN,
    "Penalty withdrawal step",
    0,
    targets.SGD_STEPS,
    limits=schedule_confidence_limits,
)
plt.show()

Delaying the penalty by only a few steps sharply reduces final sheet formation. Median sheet falls from 59% with steering from the start to less than 1% when onset is delayed to step 4. Later intervention can still lower the helix proxy (@fig-schedule-1), but the final structures contain more loop rather than recovering the sheet content achieved by early steering. Suppressing helix and promoting sheet are therefore distinct outcomes.

Later onset generally retains higher final confidence, particularly pLDDT (@fig-schedule-3), alongside the diminished sheet response. Delaying the penalty does not offer an obvious way to retain both the composition achieved by early steering and the confidence of unsteered designs.

Withdrawal tests the complementary possibility of preserving an early composition change while letting the ordinary objective guide the rest of optimization. Withdrawal at step 0 is the unsteered condition; no withdrawal is the constant-weight condition.

A single step of steering leaves the final composition close to the unsteered condition, whereas longer early exposure produces a lasting change. After 20 steps of steering, median final sheet is 55%, compared with 59% under continuous steering. Much of the composition change therefore persists through the remaining 130 steps without the penalty. This supports persistence, without establishing that the two schedules are equivalent.

Confidence does not uniformly recover after withdrawal (@fig-schedule-4). Stopping the penalty at step 20 gives a modestly higher median ipTM than continuous steering, while median binder pLDDT remains essentially unchanged. Removing the penalty after an early intervention therefore does not eliminate the confidence cost.

Together, the onset and withdrawal experiments show that early optimization has a lasting influence on the final composition. On PD-L1, steering during the first 20 steps preserves substantial sheet content after the penalty stops, whereas prolonged steering introduced only four steps later does not recover the same response. This sensitivity to optimization history does not yet explain what makes the early window decisive or establish irreversible structural basins. Nor does preserving composition after withdrawal ensure recovery of confidence, leaving the tradeoff between sheet enrichment and predicted confidence unresolved.

## Initialization also steers secondary structure {#sec-initialization}

The sensitivity to early steering raises a question about the starting point: how much of the helix preference is already present before the first sequence update? The default procedure initializes a nearly uniform soft sequence. ESMFold2 receives these probabilities alongside features derived from the most probable amino acid at each position. We therefore tested whether changing the soft probabilities could alter composition while holding that initial discrete sequence fixed.

### Soft input contributes to helicity before sequence optimization

We evaluated ten fixed 120-residue inputs on PD-L1 using their original soft probabilities, a one-hot encoding of the same sequence, and the ordinary discrete-input path. With the usual language-model feature dropout of 0.5, mean predicted helix was 57.3% for the soft inputs, 38.5% for one-hot inputs, and 39.8% for discrete inputs. There were no sequence-gradient updates in this comparison. Soft conditioning therefore contributes to the initial helix preference under these settings.

:::{.callout-note collapse="true" title="Controls for the initial-input comparison"}

The ten persisted inputs shared their discrete sequences and atom-feature tensors across representations. We used one fixed Fast-Cutoff2025 checkpoint, one internal recycle after the initial trunk pass, and 50 coordinate-sampling steps. Thus “before sequence optimization” still includes the folding model's internal processing. Two stochastic forwards were averaged within each input; the ten inputs, rather than the twenty predictions per condition, are the comparison units.

Repeating the comparison without language-model feature dropout gave mean helix fractions of 60.6%, 37.8%, and 38.6%, respectively. The input effect therefore persisted without dropout. These diagnostic runs used fixed-length inputs and compilation disabled; their absolute fractions should not be equated with the mixed-length, alternating-model design campaigns. One of 120 structures failed the backbone geometry screen and remains in the primary diagnostic summaries; it was in the soft-input, zero-dropout condition. Excluding it changed any condition mean by at most 1.21 percentage points.

This comparison identifies an input-dependent contribution to predicted helicity. It does not establish that the random discrete sequences should be nonhelical experimentally, or locate the underlying cause in the model's training.

:::

In [ ]:
# | include: false

import matplotlib.pyplot as plt
from analysis import initialization
from IPython.display import HTML, display

initialization_data = initialization.load()

### A sheet-informed soft prior redirects optimization

To retain a soft, optimizable sequence, we estimated amino-acid distributions from helix and sheet residues in the natural reference. Each distribution describes the frequency of an amino acid conditional on its secondary-structure state, excluding cysteine. We then added a small position-specific preference that kept the same amino acid most probable at each position in both conditions. The helix and sheet priors therefore started with identical hard sequences and atom features but different soft probabilities.

We ran five matched pairs for 40 updates on PD-L1, using the ordinary structural and language-model objectives with no helicity penalty. @fig-initialization-trajectories follows both the optimization proxy and secondary structure measured from saved soft states. These short trajectories use the original 150-step annealing schedule, stopped after 40 updates.

In [ ]:
# | label: fig-initialization-trajectories
# | fig-cap: "Changing the initial soft prior redirects early optimization on PD-L1. Thin lines show five trajectories per prior; thick lines show means. The two arms share each pair's initial hard sequence and atom features and use no helicity penalty. DSSP fractions come from 50-step coordinate readouts of saved soft states, which do not feed back into optimization; the distogram proxy comes from the optimization pass. Entropy measures how diffuse the amino-acid probabilities remain. Step k is before sequence update k; the last structural readout is step 39. All 400 readouts passed the geometry screen. There is no additional uniform-initialized arm in this short experiment."
# | code-fold: true
# | code-summary: "Plot code"

plt.close("all")
initialization.short_trajectories(initialization_data["short-trajectories"])
plt.show()

At the last saved state, mean helix was 86.7% with the helix prior and 33.8% with the sheet prior; mean sheet was 0% and 39.0%, respectively. All five pairs had less helix and more sheet under the sheet prior. Their mean sequence entropies were nearly identical at that step, 3.42 and 3.41 bits per residue. The difference therefore develops while both sequences remain soft, without requiring a large separation in scalar entropy.

The clearest separation emerges during optimization. At step 0, the sheet prior has lower helicity in only two of five pairs. Changing initialization need not immediately produce a sheet-rich structure to alter the later trajectory.

### Composition changes persist through full optimization, with a confidence cost

We next ran 30 full 150-step designs with the sheet prior, then tested flatter versions of that prior at initialization temperatures $T=2$ and $T=4$. Here $T=1$ is the original empirical prior; increasing $T$ moves its probabilities toward uniform. This temperature changes initialization only, and is separate from the optimizer's annealing schedule.

Each condition uses the same 30 initial hard sequences and lengths as the main PD-L1 campaign. The three prior conditions also share the actual initial atom-feature tensors. @fig-initialization-outcomes and @tbl-initialization compare their selected, refolded designs with the unsteered baseline and loss weights 1.0 and 2.0.

In [ ]:
# | label: fig-initialization-outcomes
# | fig-cap: "Final PD-L1 outcomes after 150 optimization updates, with 30 designs per condition. Each dot is one selected design evaluated by the primary critic; black bars show means and dashed lines mark the existing screen thresholds. The unsteered and loss-steered groups reuse the main campaigns. Prior T=1, T=2 and T=4 use the ordinary objective without a helicity penalty. Each tempered cohort includes its five pilot designs once."
# | code-fold: true
# | code-summary: "Plot code"

plt.close("all")
initialization.full_outcomes(initialization_data["designs"])
plt.show()

In [ ]:
# | label: tbl-initialization
# | tbl-cap: "Mean final composition and confidence on PD-L1, with 30 designs per condition. Passing requires sheet > 0.40, ipTM > 0.85 and binder pLDDT > 80, as in @tbl-screen-criteria. All measurements use the primary critic."
# | code-fold: true
# | code-summary: "Table code"

display(
    HTML(
        initialization.summary(initialization_data["designs"])
        .style.format(
            {
                "Mean helix (%)": "{:.1f}",
                "Mean sheet (%)": "{:.1f}",
                "Mean ipTM": "{:.3f}",
                "Mean pLDDT": "{:.1f}",
                "Screen passes / 30": "{:.0f}",
            }
        )
        .to_html()
    )
)

The original sheet prior increases mean final sheet from 11.5% to 42.0%, but mean ipTM falls from 0.892 to 0.525 and binder pLDDT from 85.2 to 66.8. Changing the starting probabilities therefore has a lasting effect under the ordinary objective, accompanied by substantially lower predicted confidence.

Tempering partly recovers confidence while reducing the composition shift. At $T=2$ and $T=4$, mean sheet is 34.3% and 27.7%, mean ipTM is 0.734 and 0.726, and mean pLDDT is 73.5 and 77.1. The median first step below two bits of sequence entropy moves from 76 at $T=1$ to 90 at $T=2$ and 97.5 at $T=4$, approaching 101.5 without steering.

The performance comparison depends on the metric. Tempered initialization has higher mean pLDDT than loss weight 2.0, but lower mean ipTM, less sheet, and fewer candidates passing the combined screen. The three prior conditions yield 0, 1, and 1 passing candidates, compared with 4 at loss weight 1.0 and 9 at weight 2.0. Most high-confidence prior-initialized designs remain helical: eight of nine $T=4$ designs passing both confidence thresholds have zero sheet.

These results establish initialization as a second control on the composition distribution. They do not show an advantage over loss steering under the selected screen. The short matched-prior experiment suggests that the identities represented in the soft sequence matter beyond entropy alone; the temperature experiment changes composition, probability contrast, and entropy together, so it does not isolate which of those properties explains confidence recovery.

:::{.callout-note collapse="true" title="Constructing the priors and interpreting the follow-up"}

We pooled amino-acid counts over residues assigned helix or sheet by the same three-state PyDSSP classifier used elsewhere in this notebook. Cysteine and noncanonical residues were excluded before normalization. These are empirical conditional frequencies, not context-independent folding propensities.

For a prior $q$ over 19 amino acids and the desired starting identity $a_i$ at position $i$, we set

$$
p_i(a)=q(a)+b\left[\mathbf 1(a=a_i)-\frac{1}{19}\right].
$$

In the 40-step helix/sheet comparison, $b=0.12$ for both priors. Each pair uses the same uniformly sampled $a_i$ values and the same saved atom features. The boost exceeds the range of either prior, so $a_i$ remains the unique most probable identity at every position, while all 19 probabilities stay positive.

For the full sheet-prior comparison, we use the archived initial hard sequences from the main campaign and define

$$
q_T(a)=\frac{q(a)^{1/T}}{\sum_c q(c)^{1/T}},
\qquad
b_T=0.12\,\frac{\max(q_T)-\min(q_T)}{\max(q)-\min(q)}.
$$

Mutable logits are initialized to the optimizer's step-0 temperature times $\log p_i$; the prior is applied once and is not restored after updates. All binder positions remain mutable. The full campaigns share the initial atom tensors across prior conditions; the historical unsteered and loss-steered campaigns did not archive those tensors, so their matching is limited to initial hard sequence and length.

Optimization uses one coordinate-solver step per update, including late confidence selection. The 40-step experiment obtains separate 50-step coordinate readouts from saved soft states. Full designs use the existing sequence-selection rule and four final critics with 200 solver steps and three loops; the main results retain the same primary critic as the rest of the paper. All 360 final critic structures from the three prior cohorts passed the backbone geometry checks.

This follow-up was developed adaptively on one target. The five-seed $T=2$ and $T=4$ pilots were extended to 30 seeds after inspection and remain part of those cohorts. Seeds identify matched starting inputs but do not make numerical execution reproducible. Neither repeated trajectory frames nor predictions from different critics are independent designs. The 40-step pairs are separate runs from the full campaigns.

Across all four critics, tempering improves mean ipTM and pLDDT relative to $T=1$. The ipTM ranking of $T=2$ versus $T=4$ varies by critic, whereas all four give $T=4$ higher mean pLDDT. The follow-up supports an initialization effect and a confidence tradeoff, without establishing an optimal temperature or experimental binder quality.


In [ ]:
# | code-fold: true
# | code-summary: "Recalculate priors from the reference structures"

from analysis import priors

initialization_prior = priors.from_reference(natural)
print(
    f"Eligible reference residues: "
    f"{initialization_prior['residue_counts']['helix']:,} helix; "
    f"{initialization_prior['residue_counts']['sheet']:,} sheet."
)
for prior_temperature in (1, 2, 4):
    _, prior_boost = priors.tempered_sheet(initialization_prior, prior_temperature)
    print(f"T = {prior_temperature}: b = {prior_boost:.6f}")

:::

## Takeaways

Adding a local distance penalty changes the predicted secondary-structure composition of ESMFold2 binder designs across five targets. Despite lower average confidence, steering increases the number of candidates meeting a sheet-content and confidence screen. Weight 2.0 gives the largest pooled count, while the most productive weight varies by target.

Initialization provides another control on PD-L1: changing the soft amino-acid probabilities can shift final composition under the original objective. Tempering the sheet prior partly recovers predicted confidence, but produces fewer candidates meeting the combined screen than explicit loss steering.

### What the experiments establish

An added objective can redirect the output of a pretrained binder-design workflow. Across five targets, increasing the helicity penalty shifts predicted structures toward sheet and loop, with lower confidence scores. Under the selected sheet and confidence criteria, weight 2.0 yields 29 candidates compared with 15 without steering (@tbl-sheet-screen). The per-target counts show why the choice of weight should also reflect the target.

On PD-L1, early exposure is much more effective at producing sheet than late intervention, and much of the effect persists after withdrawal. Both the weight and timing of an objective can therefore shape the candidate pool without changing model weights.

The initialization experiments extend that control to the starting sequence probabilities. The same initial hard sequence can follow different composition trajectories under different soft priors, and the effects can persist through 150 updates. Soft conditioning contributes to the initial helix preference under the tested settings, while subsequent optimization strengthens or redirects that response. This identifies an accessible control before sequence optimization without establishing a single cause of the model's structural preferences.

Loss steering currently gives the larger yield under our combined screen. Initialization remains informative about how the procedure generates its composition distribution, even though it does not improve that yield. Secondary-structure composition is one of many properties a design campaign might seek to control. These results illustrate how changing objectives and initialization can customize ESMFold2's design procedure after training; whether similar interventions can control other properties remains to be tested. The experimental usefulness and fold diversity of the resulting candidates remain open questions.

### The composition proxy leaves important properties unconstrained

A short-distance penalty can remove helices without specifying a complete alternative fold. It does not reward strand pairing, sheet closure, packing, or a particular binding interface. An optimizer can lower the penalty by producing extended or irregular structure, which is consistent with the increase in loop as well as sheet.

Secondary-structure fractions therefore need to be paired with structural inspection. A β-rich structure may be an open sheet with exposed edges, a compact sheet-containing fold, or a poorly packed arrangement. The present composition measurements do not quantify those alternatives, so they cannot support a claim of increased fold diversity or improved developability.

### The biological meaning of lower confidence remains unresolved

Both loss steering and sheet-informed initialization produce more β-rich candidates with lower average confidence than the unsteered baseline. Their relative performance depends on the metric: tempering improves pLDDT more consistently than ipTM. These score changes leave the central biological question unresolved. The scores may correctly identify structures that fold or bind poorly. Alternatively, the model may assign lower confidence to viable β-rich designs because they fall outside the structural compositions it most readily generates.

The strong helical preference of the unsteered workflow makes this possibility worth investigating. It establishes a bias in the generated composition distribution, but does not establish whether confidence is miscalibrated: a preference during design and an inaccurate assessment of a finished design are different observations. The practical consequence is substantial. If lower scores reflect weaker binders, steering carries a biological cost; if they underestimate viable β-rich candidates, confidence filtering could discard useful designs.

Experimental evaluation of the steered candidates is needed to determine what these score changes mean for binder design.

## Next steps

Our main next step is to test whether the β-rich candidates produced by loss steering and sheet-informed initialization are viable in the laboratory. We plan to evaluate whether these sequences yield folded proteins that bind their intended targets. This will establish whether expanding the predicted composition distribution also expands the pool of useful binders.

Those results will also help interpret the lower computational confidence of steered designs. The central question is whether the scores reflect a real loss of candidate quality, underestimate successful designs with different structural compositions, or capture some combination of the two.